# Обработка аудио в Python
Нейрокомпьютеры. Практика 1.

In [1]:
#@title Проверить библиотеки { display-mode: "form" }
# Ставим недостающие пакеты одним вызовом pip.
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
requirements = {
    "gradio": "gradio==6.5.1",
    "librosa": "librosa==0.11.0",
    "soundfile": "soundfile==0.13.1",
    "matplotlib": "matplotlib==3.11.1", # Добавил от себя
    "numpy": "numpy==2.5.3", # Добавил от себя
}
missing = []
for name, requirement in requirements.items():
    try:
        version(name)
    except PackageNotFoundError:
        missing.append(requirement)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
print("Python", sys.version.split()[0])
for name in ("numpy", "matplotlib", "librosa", "soundfile", "gradio"):
    print(name, version(name))

Python 3.13.2
numpy 2.5.3
matplotlib 3.11.1
librosa 0.11.0
soundfile 0.13.1
gradio 6.5.1


## Подготовка
NumPy нужен для работы с массивами, soundfile читает и сохраняет аудио, Matplotlib строит графики. Librosa меняет частоту дискретизации и вычисляет частотное представление сигнала.

In [ ]:
from pathlib import Path
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import librosa
import librosa.display

In [ ]:
#@title Скачать записи { display-mode: "form", run: "auto" }
#@markdown Прямые ссылки на два аудиофайла из Gist.
voice_url = "https://gist.github.com/Doster-d/9d9adbcf8e3df2112fc196b52eae74f0/raw/82f2cee5f5ef19cfa792533c0cc018f8e7f17be8/nk-p01-dan-golos.mp3" #@param {type:"string"}
melody_url = "https://gist.github.com/Doster-d/9d9adbcf8e3df2112fc196b52eae74f0/raw/82f2cee5f5ef19cfa792533c0cc018f8e7f17be8/nk-p01-dan-melodiya.wav" #@param {type:"string"}
data_folder = "nk-p01-data" #@param {type:"string"}
refresh_files = True #@param {type:"boolean"}

import hashlib
import json
import socket
import time as clock
from tempfile import NamedTemporaryFile
from http.client import IncompleteRead
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

DATA_DIR = Path(data_folder).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR = Path.cwd() / "nk-p01-results"
RESULT_DIR.mkdir(exist_ok=True)
MAX_AUDIO_BYTES = 64 * 1024 * 1024
AUDIO_SUFFIXES = {
    "WAV": ".wav",
    "WAVEX": ".wav",
    "RF64": ".wav",
    "MP3": ".mp3",
    "FLAC": ".flac",
    "OGG": ".ogg",
    "AIFF": ".aiff"
}


def audio_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def checked_audio(path: Path):
    info = sf.info(path)
    if info.frames < 1 or info.samplerate <= 0 or info.channels < 1:
        raise ValueError("В записи нет звуковых отсчетов.")
    if info.format not in AUDIO_SUFFIXES:
        raise ValueError("Нужен аудиофайл WAV, MP3, FLAC, OGG или AIFF.")
    if info.frames * info.channels > 32_000_000:
        raise ValueError("Запись слишком длинная для этих опытов.")
    samples, _ = sf.read(path, dtype="float32", always_2d=True)
    if len(samples) != info.frames or not np.isfinite(samples).all():
        raise ValueError("Не удалось прочитать все отсчеты записи без ошибок.")
    return info


def prepare_audio(url: str, stem: str, label: str, field: str) -> Path:
    url = url.strip()
    record_path = DATA_DIR / f"{stem}.json"
    try:
        record = json.loads(record_path.read_text(encoding="utf-8"))
        if not isinstance(record, dict):
            record = {}
    except (OSError, ValueError):
        record = {}
    cached = DATA_DIR / Path(str(record.get("filename", "missing"))).name
    if not url:
        candidates = sorted(p for p in DATA_DIR.glob(stem + ".*")
                            if p.suffix.lower() in AUDIO_SUFFIXES.values())
        if len(candidates) != 1:
            raise ValueError(f"{label}: заполни поле {field} в ячейке «Скачать записи». "
                             f"Для ручной загрузки оставь один файл {stem}.mp3 или {stem}.wav "
                             f"в папке {DATA_DIR}. Сейчас найдено: {len(candidates)}.")
        checked_audio(candidates[0])
        print(f"{label}: используется загруженный вручную файл; ссылка не задана.")
        return candidates[0]
    parsed = urlparse(url)
    if parsed.scheme not in {"https", "http"} or not parsed.hostname:
        raise ValueError(f"{label}: в {field} нужна полная прямая ссылка на аудиофайл.")
    if parsed.hostname == "gist.github.com" and "/raw" not in parsed.path:
        raise ValueError(f"{label}: это страница Gist. Скопируй ссылку Raw на сам файл "
                         "или адрес аудиовложения.")
    if (not refresh_files and record.get("url") == url and cached.is_file()
            and record.get("sha256") == audio_sha256(cached)):
        try:
            checked_audio(cached)
            print(f"{label}: проверенный файл уже скачан.")
            return cached
        except (OSError, RuntimeError, ValueError):
            pass
    temp_path = None
    try:
        request = Request(url, headers={"User-Agent": "nk-p01-audio-loader"})
        started = clock.monotonic()
        with urlopen(request, timeout=30) as response:
            content_type = response.headers.get_content_type()
            if content_type in {"text/html", "application/xhtml+xml", "application/json"}:
                raise ValueError("Ссылка возвращает страницу или текст, а не аудиофайл. "
                                 "Проверь Raw или адрес вложения.")
            expected = int(response.headers.get("Content-Length") or 0)
            if expected > MAX_AUDIO_BYTES:
                raise ValueError("Файл больше 64 МиБ; для пары нужен короткий фрагмент.")
            total = 0
            with NamedTemporaryFile(dir=DATA_DIR, prefix=stem + "-", suffix=".part",
                                    delete=False) as output:
                temp_path = Path(output.name)
                while chunk := response.read(1024 * 1024):
                    total += len(chunk)
                    if total > MAX_AUDIO_BYTES or clock.monotonic() - started > 120:
                        raise ValueError("Загрузка слишком долгая или файл больше 64 МиБ.")
                    output.write(chunk)
            if not total or (expected and total != expected):
                raise ValueError("Файл скачался не полностью; запусти ячейку еще раз.")
        info = checked_audio(temp_path)
        filename = stem + AUDIO_SUFFIXES[info.format]
        destination = DATA_DIR / filename
        checksum = audio_sha256(temp_path)
        temp_path.replace(destination)
        record_path.write_text(json.dumps({"url": url, "filename": filename,
                                          "sha256": checksum}, ensure_ascii=False), encoding="utf-8")
        print(f"{label}: скачан {filename}.")
        return destination
    except HTTPError as error:
        raise RuntimeError(f"{label}: сервер ответил HTTP {error.code}. Проверь доступ по ссылке "
                           "без входа в аккаунт и запусти ячейку еще раз.") from None
    except (URLError, socket.timeout, TimeoutError, IncompleteRead) as error:
        raise RuntimeError(f"{label}: не удалось скачать файл. Проверь ссылку и сеть. "
                           f"Можно загрузить {stem}.mp3 или {stem}.wav вручную в {DATA_DIR} "
                           f"и очистить поле {field}.") from None
    except (OSError, RuntimeError, ValueError) as error:
        raise RuntimeError(f"{label}: файл не подготовлен. {error}") from None
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)


# Обе записи обязательны. Речь не подменяется мелодией при ошибке загрузки.
paths = {}
for label, link, stem, field in [
    ("Речь", voice_url, "nk-p01-dan-golos", "voice_url"),
    ("Мелодия", melody_url, "nk-p01-dan-melodiya", "melody_url"),
]:
    path = prepare_audio(link, stem, label, field)
    info = sf.info(path)
    paths[label] = path
    print(f"  {info.duration:.1f} с; {info.samplerate:,} Гц; каналов: {info.channels}")
print("Файлы готовы.")

Речь: скачан nk-p01-dan-golos.mp3.
  31.5 с; 24,000 Гц; каналов: 1
Мелодия: скачан nk-p01-dan-melodiya.wav.
  5.0 с; 44,100 Гц; каналов: 1
Файлы готовы.


In [ ]:
#@title Подготовить Gradio для прослушивания { display-mode: "form" }
# Только выдача WAV и интерфейс; учебных алгоритмов здесь нет.
import os
import html
from urllib.parse import urlencode
from IPython.display import display, HTML
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import gradio as gr

AUDIO_DIR = RESULT_DIR / "players"
AUDIO_DIR.mkdir(exist_ok=True)
PLAY_GAIN = 0.35
AUDIO_VIEWS = globals().get("AUDIO_VIEWS", {})
AUDIO_URL = None


def listen(title, items):
    """Сохраняет готовые массивы с единым усилением и показывает проигрыватели."""
    if not 1 <= len(items) <= 3:
        raise ValueError("Для сравнения нужны от одной до трех записей.")
    digest = hashlib.sha256((title + str(PLAY_GAIN)).encode())
    ready = []
    for label, samples, rate in items:
        samples = np.asarray(samples, dtype=np.float32)
        if samples.ndim != 1 or not samples.size or not np.isfinite(samples).all():
            raise ValueError("Нужен непустой монофонический массив без пропусков.")
        if int(rate) <= 0:
            raise ValueError("Частота должна быть положительной.")
        if float(np.max(np.abs(samples))) * PLAY_GAIN >= 0.99:
            raise ValueError("Слишком большой уровень: верни настройки опыта в указанный диапазон.")
        digest.update(str((label, int(rate))).encode())
        digest.update(samples.tobytes())
        ready.append((label, samples, int(rate)))
    view_id = digest.hexdigest()[:24]
    folder = AUDIO_DIR / view_id
    folder.mkdir(exist_ok=True)
    paths_for_view = []
    for index, (label, samples, rate) in enumerate(ready, 1):
        path = folder / f"nk-p01-dan-result-{index}.wav"
        if not path.exists():
            sf.write(path, PLAY_GAIN * samples, rate, subtype="PCM_16")
        paths_for_view.append((label, str(path.resolve())))
    AUDIO_VIEWS[view_id] = (title, paths_for_view)
    if not AUDIO_URL:
        print("Gradio не запущен. Расчет готов; повтори запуск проигрывателя, затем этот опыт.")
        return
    url = AUDIO_URL.rstrip("/") + "/?" + urlencode({"view": view_id, "__theme": "light"})
    safe_url = html.escape(url, quote=True)
    display(HTML(f'<iframe title="{html.escape(title, quote=True)}" loading="lazy" '
                 f'src="{safe_url}" width="100%" height="350" style="border:0" allow="autoplay"></iframe>'
                 f'<p><a href="{safe_url}" target="_blank" rel="noopener">'
                 'Открыть это сравнение в Gradio</a></p>'))


def load_audio_view(view_id):
    entry = AUDIO_VIEWS.get(str(view_id))
    if entry is None:
        return ["Открой проигрыватель под выполненной ячейкой опыта."] + [gr.update(visible=False)] * 3
    title, paths_for_view = entry
    updates = ["### " + title]
    for index in range(3):
        if index < len(paths_for_view):
            label, path = paths_for_view[index]
            updates.append(gr.update(value=path, label=label, visible=True))
        else:
            updates.append(gr.update(value=None, visible=False))
    return updates


old_app = globals().get("audio_app")
if old_app is not None and getattr(old_app, "is_running", False):
    old_app.close()
with gr.Blocks(title="НК П1. Звук", analytics_enabled=False) as audio_app:
    heading = gr.Markdown("Загрузка звука...")
    view_input = gr.Textbox(visible=False)
    with gr.Row():
        players = [gr.Audio(type="filepath", interactive=False, editable=False,
                           autoplay=False, buttons=["download"], min_width=210)
                   for _ in range(3)]
    audio_app.load(load_audio_view, inputs=[view_input], outputs=[heading, *players],
                   js="() => [new URLSearchParams(window.location.search).get('view') || '']",
                   queue=False, api_visibility="private")
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False
try:
    _, local_url, share_url = audio_app.launch(
        share=is_colab, inline=False, inbrowser=False, prevent_thread_lock=True,
        quiet=True, show_error=True, allowed_paths=[str(AUDIO_DIR.resolve())],
        footer_links=[], css=".gradio-container { padding: 8px !important; }")
    AUDIO_URL = share_url if is_colab else local_url
    print("Gradio готов. Проигрыватели появятся рядом с опытами.")
except Exception as error:
    print("Не удалось запустить Gradio:", error)
    print("Повтори эту ячейку. Расчеты и графики от запуска проигрывателя не зависят.")

Closing server running on port: 7860
* Running on public URL: https://37346ef7dd6cc36bbc.gradio.live
Gradio готов. Проигрыватели появятся рядом с опытами.


### Исходные записи
`sf.read` возвращает массив и частоту дискретизации. Стереозапись содержит два столбца; здесь усредняем каналы и дальше работаем с одним. Словарь `recordings` хранит обе записи под названиями «Речь» и «Мелодия».

In [ ]:
recordings = {}
for name, path in paths.items():
    samples, rate = sf.read(path, dtype="float32", always_2d=True)
    recordings[name] = (samples.mean(axis=1), rate)

voice, voice_sr = recordings["Речь"]
melody, melody_sr = recordings["Мелодия"]
listen("Исходные записи целиком", [
    ("Речь", voice, voice_sr),
    ("Мелодия", melody, melody_sr),
])

## 1. Звук как массив
В цифровой записи звук представлен последовательностью чисел. Одно число называется отсчетом. Частота дискретизации `sr` показывает, сколько отсчетов приходится на секунду.

`signal[:12]` берет первые двенадцать чисел. `len(signal) / sr` дает длительность записи в секундах.

In [ ]:
#@title Числа в записи { display-mode: "both", run: "auto" }
recording = "Мелодия" #@param ["Речь", "Мелодия"]

signal, sr = recordings[recording]
print("Первые 12 отсчетов:", signal[:12])
print("Число отсчетов:", len(signal))
print("Тип чисел:", signal.dtype)
print("Частота дискретизации:", sr, "Гц")
print("Длительность:", round(len(signal) / sr, 2), "с")

Первые 12 отсчетов: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Число отсчетов: 220496
Тип чисел: float32
Частота дискретизации: 44100 Гц
Длительность: 5.0 с


### Временной график
По горизонтали отложено время, по вертикали амплитуда. Чтобы получить время каждого отсчета, делим его номер на `sr`. Знак амплитуды показывает направление колебания; отрицательное значение не означает отсутствие звука.

In [ ]:
#@title График записи { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]

signal, sr = recordings[recording]
time = np.arange(len(signal)) / sr
plt.figure(figsize=(10, 2.8))
plt.plot(time, signal, linewidth=0.6)
plt.xlabel("Время, с")
plt.ylabel("Амплитуда")
plt.title(recording)
plt.tight_layout()
plt.show()

### Отдельные отсчеты
На общем графике точки сливаются в линию. Увеличим десять миллисекунд: точки обозначают отсчеты, линии между ними помогают рассмотреть форму сигнала. Поле `position` задает положение участка в процентах от длины записи.

In [ ]:
#@title Увеличение участка { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
position = 72 #@param {type:"slider", min:0, max:100, step:1}

signal, sr = recordings[recording]
count = max(1, round(0.01 * sr))
left = round(max(0, len(signal) - count) * position / 100)
right = min(left + count, len(signal))
time_ms = np.arange(left, right) / sr * 1000
plt.figure(figsize=(10, 2.8))
plt.plot(time_ms, signal[left:right], ".-", linewidth=0.7)
plt.xlabel("Время, мс")
plt.ylabel("Амплитуда")
plt.title(f"Участок с {left / sr:.3f} с")
plt.tight_layout()
plt.show()

### Срез и обратное воспроизведение
`signal[start:stop]` берет часть массива: отсчет с номером `start` входит в нее, отсчет `stop` уже не входит. Секунды переводим в номера отсчетов умножением на `sr`.

`fragment[::-1]` меняет порядок чисел на обратный. Число отсчетов и частота остаются прежними, поэтому длительность не меняется.

In [ ]:
#@title Фрагмент вперед и назад { display-mode: "both", run: "auto" }
recording = "Мелодия" #@param ["Речь", "Мелодия"]
start_s = 0.0 #@param {type:"slider", min:0, max:40, step:0.1}
duration_s = 3.0 #@param {type:"slider", min:0.5, max:10, step:0.5}

signal, sr = recordings[recording]
# Ограничиваем границы длиной записи; ниже выводим фактический интервал.
start = min(round(start_s * sr), max(0, len(signal) - sr // 2))
stop = min(start + round(duration_s * sr), len(signal))
fragment = signal[start:stop].copy()
backwards = fragment[::-1]
print(f"Границы: {start / sr:.2f}–{stop / sr:.2f} с")
print("Отсчетов вперед и назад:", len(fragment), len(backwards))
print("Два разворота возвращают исходник:", np.array_equal(backwards[::-1], fragment))
listen("Фрагмент вперед и назад", [
    ("Обычное воспроизведение", fragment, sr),
    ("Обратное воспроизведение", backwards, sr),
])

Границы: 0.00–3.00 с
Отсчетов вперед и назад: 132300 132300
Два разворота возвращают исходник: True


### Сохранение WAV
`sf.write` записывает массив в файл вместе с частотой дискретизации. Здесь `FLOAT` сохраняет дробные значения массива. Форма ниже сохраняет обычную и обратную версии выбранного в ней фрагмента. Исходные файлы из Gist не меняются.

In [ ]:
#@title Сохранить фрагмент { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
start_s = 0.0 #@param {type:"slider", min:0, max:40, step:0.1}
duration_s = 5.5 #@param {type:"slider", min:0.5, max:10, step:0.5}

signal, sr = recordings[recording]
start = min(round(start_s * sr), max(0, len(signal) - sr // 2))
stop = min(start + round(duration_s * sr), len(signal))
fragment = signal[start:stop].copy()
result_path = RESULT_DIR / "nk-p01-dan-fragment.wav"
reverse_path = RESULT_DIR / "nk-p01-dan-fragment-reversed.wav"
sf.write(result_path, fragment, sr, subtype="FLOAT")
sf.write(reverse_path, fragment[::-1], sr, subtype="FLOAT")
saved, saved_sr = sf.read(result_path, dtype="float32")
print(f"Границы: {start / sr:.2f}–{stop / sr:.2f} с")
print("Файлы:", result_path.name, reverse_path.name)
print("Частота и значения совпали:", saved_sr == sr and np.array_equal(saved, fragment))
listen("Сохраненный фрагмент", [("WAV", saved, saved_sr)])

Границы: 0.00–5.50 с
Файлы: nk-p01-dan-fragment.wav nk-p01-dan-fragment-reversed.wav
Частота и значения совпали: True


### Соединение массивов
`np.concatenate` соединяет массивы последовательно. Разделим фрагмент на две части и поставим вторую перед первой. Внутри частей порядок отсчетов сохранится.

В этом и следующих опытах с записями используем первые восемь секунд. Если запись короче, берем ее целиком.

In [ ]:
#@title Перестановка частей { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
split_percent = 50 #@param {type:"slider", min:10, max:90, step:5}

signal, sr = recordings[recording]
fragment = signal[:8 * sr]
middle = round(len(fragment) * split_percent / 100)
first = fragment[:middle]
second = fragment[middle:]
# let's reverse second part!
swapped = np.concatenate([second[::-1], first])
print("Граница частей:", round(middle / sr, 2), "с")
listen("Перестановка частей", [
    ("Исходный порядок", fragment, sr),
    ("Вторая часть, затем первая", swapped, sr),
])

Граница частей: 4.0 с


## 2. Частота и точность представления
### Синусоидальный тон
Синусоида описывает повторяющееся колебание. Частота `frequency` задает число колебаний в секунду, амплитуда `amplitude` задает максимальное отклонение от нуля. В форме меняются эти два параметра; частота дискретизации остается 44 100 Гц.

In [ ]:
#@title Синусоида { display-mode: "both", run: "auto" }
frequency = 880 #@param [220, 440, 880] {type:"raw"}
amplitude = 0.3 #@param {type:"slider", min:0.05, max:0.5, step:0.05}

tone_sr = 44_100
time = np.arange(2 * tone_sr) / tone_sr
tone = amplitude * np.sin(2 * np.pi * frequency * time)
plt.figure(figsize=(10, 2.6))
plt.plot(time[:441] * 1000, tone[:441])
plt.ylim(-0.55, 0.55)
plt.xlabel("Время, мс")
plt.ylabel("Амплитуда")
plt.title(f"Частота колебания {frequency} Гц")
plt.tight_layout()
plt.show()
listen("Синусоидальный тон", [(f"{frequency} Гц", tone, tone_sr)])

### Сумма двух тонов
Массивы одинаковой длины можно сложить: складываются значения на одинаковых позициях. Первый тон имеет частоту 440 Гц, частоту второго меняет форма. На временном графике видна сумма колебаний.

In [ ]:
#@title Два тона { display-mode: "both", run: "auto" }
second_frequency = 880 #@param [220, 440, 660, 880, 1100] {type:"raw"}

mix_sr = 44_100
time = np.arange(2 * mix_sr) / mix_sr
first_tone = 0.15 * np.sin(2 * np.pi * 440 * time)
second_tone = 0.15 * np.sin(2 * np.pi * second_frequency * time)
mixed = first_tone + second_tone
plt.figure(figsize=(10, 2.6))
plt.plot(time[:441] * 1000, mixed[:441])
plt.xlabel("Время, мс")
plt.ylabel("Амплитуда")
plt.title("Сумма двух синусоид")
plt.tight_layout()
plt.show()
listen("Два тона и их сумма", [
    ("440 Гц", first_tone, mix_sr),
    (f"{second_frequency} Гц", second_tone, mix_sr),
    ("Сумма", mixed, mix_sr),
])

### Скорость воспроизведения
Проигрыватель получает массив и частоту: сколько отсчетов нужно воспроизвести за секунду. Если передать тот же массив с удвоенной частотой, запись закончится вдвое быстрее. При таком ускорении повысится и высота звука.

In [ ]:
#@title Скорость записи { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
speed = 1.5 #@param {type:"slider", min:0.5, max:2, step:0.25}

signal, sr = recordings[recording]
fragment = signal[:8 * sr]
play_sr = round(sr * speed)
print("Отсчетов в обоих вариантах:", len(fragment))
print(f"Длительность: {len(fragment) / sr:.2f} и {len(fragment) / play_sr:.2f} с")
listen("Изменение скорости", [
    ("Исходник", fragment, sr),
    (f"Скорость x{speed:g}", fragment, play_sr),
])

Отсчетов в обоих вариантах: 192000
Длительность: 8.00 и 5.33 с


### Изменение частоты дискретизации
`librosa.resample` пересчитывает массив для новой частоты. Число отсчетов меняется, длительность сохраняется с точностью до округления. При уменьшении частоты отбрасываются высокочастотные компоненты, которые нельзя представить при новой частоте дискретизации.

На графике показан один и тот же короткий интервал; точки обозначают отсчеты.

In [ ]:
#@title Частота дискретизации { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
target_sr: int = 4000 #@param [4000, 8000, 16000, 22050, 44100] {type:"raw"}

signal, sr = recordings[recording]
fragment = signal[:8 * sr]
resampled = librosa.resample(fragment, orig_sr=sr, target_sr=target_sr)
print("Число отсчетов:", len(fragment), "и", len(resampled))
print(f"Длительность: {len(fragment) / sr:.5f} и {len(resampled) / target_sr:.5f} с")
# Выбираем участок около максимальной амплитуды.
zoom_start = max(0, np.argmax(np.abs(fragment)) / sr - 0.004)
plt.figure(figsize=(10, 2.8))
for label, values, rate in [("Исходник", fragment, sr), ("Пересчет", resampled, target_sr)]:
    positions = np.arange(len(values)) / rate
    keep = (positions >= zoom_start) & (positions < zoom_start + 0.008)
    plt.plot(positions[keep] * 1000, values[keep], ".-", label=label)
plt.xlabel("Время, мс")
plt.ylabel("Амплитуда")
plt.legend()
plt.tight_layout()
plt.show()
listen("Изменение частоты дискретизации", [
    (f"Исходник: {sr} Гц", fragment, sr),
    (f"Результат: {target_sr} Гц", resampled, target_sr),
])

Число отсчетов: 192000 и 32000
Длительность: 8.00000 и 8.00000 с


Длительность определяется числом отсчетов $N$ и частотой дискретизации $f_s$:

$$T = \frac{N}{f_s}.$$

При ускорении меняется только частота воспроизведения. При пересчете вместе с частотой меняется число отсчетов. Увеличение частоты не возвращает ранее потерянные детали записи.

### Квантование
Квантование заменяет амплитуды ближайшими допустимыми значениями. При разрядности `bits` доступно `2 ** bits` уровней: например, при двух битах их четыре. Частота дискретизации и число отсчетов в этом опыте не меняются.

`np.round` округляет, `np.clip` ограничивает коды выбранным диапазоном. На увеличенном графике видна ошибка округления.

In [ ]:
#@title Разрядность { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
bits = 16 #@param [2, 4, 8, 16] {type:"raw"}

signal, sr = recordings[recording]
fragment = signal[:8 * sr]
scale = 2 ** (bits - 1)
codes = np.round(fragment * scale)
codes = np.clip(codes, -scale, scale - 1)
quantized = codes / scale
print("Допустимых уровней:", 2 ** bits)
print("Максимальная ошибка:", round(float(np.max(np.abs(fragment - quantized))), 5))
left = max(0, int(np.argmax(np.abs(fragment))) - round(0.004 * sr))
right = min(left + round(0.008 * sr), len(fragment))
time_ms = np.arange(left, right) / sr * 1000
plt.figure(figsize=(10, 2.8))
plt.plot(time_ms, fragment[left:right], label="Исходник")
plt.step(time_ms, quantized[left:right], where="mid", label=f"{bits} бит")
plt.xlabel("Время, мс")
plt.ylabel("Амплитуда")
plt.legend()
plt.tight_layout()
plt.show()
listen("Квантование", [
    ("Исходник", fragment, sr),
    (f"{bits} бит", quantized, sr),
])

Допустимых уровней: 65536
Максимальная ошибка: 2e-05


### Объем несжатых данных
Число отсчетов умножаем на число бит в одном отсчете и делим на восемь, чтобы получить байты.

In [ ]:
#@title Расчет объема { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
bits = 8 #@param [2, 4, 8, 16] {type:"raw"}

signal, sr = recordings[recording]
fragment = signal[:8 * sr]
size_bytes = int(np.ceil(len(fragment) * bits / 8))
print("Число отсчетов:", len(fragment))
print(f"При {bits} битах: {size_bytes / 1000:.2f} кБ")
print(f"При 16 битах: {len(fragment) * 2 / 1000:.2f} кБ")

Число отсчетов: 192000
При 8 битах: 192.00 кБ
При 16 битах: 384.00 кБ


## 3. Шум и частотное представление
### Добавление шума

Давайте к каждому отсчету прибавим случайное число.
Параметр `noise_std` задает величину разброса этих чисел.
Начальное состояние генератора `42` позволяет при повторном запуске получить тот же шум.

In [ ]:
#@title Запись с шумом { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
noise_std = 0.04 #@param {type:"slider", min:0, max:0.12, step:0.01}

signal, sr = recordings[recording]
fragment = signal[:8 * sr]
rng = np.random.default_rng(42)
noise = rng.normal(0, noise_std, len(fragment))
noisy = fragment + noise
time = np.arange(len(fragment)) / sr
plt.figure(figsize=(10, 2.8))
plt.plot(time, noisy, label="С шумом", linewidth=0.6)
plt.plot(time, fragment, label="Исходник", linewidth=0.6)
plt.xlabel("Время, с")
plt.ylabel("Амплитуда")
plt.legend()
plt.tight_layout()
plt.show()
listen("Добавление шума", [("Исходник", fragment, sr), ("С шумом", noisy, sr)])

### Спектр двух тонов
Спектр показывает частотные составляющие сигнала. У суммы тонов с разными частотами будут два пика. Если частоты совпадут, колебания сложатся в один тон.

`np.fft.rfft` вычисляет частотное представление всей записи, `np.abs` берет модули коэффициентов.

In [ ]:
#@title Частоты в сумме тонов { display-mode: "both", run: "auto" }
second_frequency = 660 #@param [220, 440, 660, 880, 1100] {type:"raw"}

tone_sr = 44_100
time = np.arange(2 * tone_sr) / tone_sr
mixed = 0.15 * np.sin(2 * np.pi * 440 * time)
mixed += 0.15 * np.sin(2 * np.pi * second_frequency * time)
frequencies = np.fft.rfftfreq(len(mixed), d=1 / tone_sr)
spectrum = np.abs(np.fft.rfft(mixed)) / len(mixed)
plt.figure(figsize=(10, 2.8))
plt.plot(frequencies, spectrum)
plt.xlim(0, 1400)
plt.xlabel("Частота, Гц")
plt.ylabel("Модуль / число отсчетов")
plt.title("Спектр суммы тонов")
plt.tight_layout()
plt.show()
listen("Сумма тонов", [(f"440 и {second_frequency} Гц", mixed, tone_sr)])

### Спектрограмма
Для речи и мелодии состав частот меняется со временем. Кратковременное преобразование Фурье (`stft`) вычисляет коэффициенты для последовательных перекрывающихся участков. `n_fft` задает число отсчетов в участке, `hop_length` задает шаг.

На спектрограмме по горизонтали время, по вертикали частота, цвет показывает уровень в децибелах. У двух рисунков одинаковая шкала и общий опорный уровень, поэтому их можно сравнивать.

In [ ]:
#@title Спектрограммы исходника и шума { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
noise_std = 0.04 #@param {type:"slider", min:0, max:0.12, step:0.01}
n_fft = 1024 #@param [512, 1024, 2048] {type:"raw"}

signal, sr = recordings[recording]
fragment = signal[:8 * sr]
noisy = fragment + np.random.default_rng(42).normal(0, noise_std, len(fragment))
hop_length = n_fft // 4
D_clean = librosa.stft(fragment, n_fft=n_fft, hop_length=hop_length)
D_noise = librosa.stft(noisy, n_fft=n_fft, hop_length=hop_length)
reference = max(np.abs(D_clean).max(), np.abs(D_noise).max()) or 1.0
clean_db = librosa.amplitude_to_db(np.abs(D_clean), ref=reference, top_db=None)
noise_db = librosa.amplitude_to_db(np.abs(D_noise), ref=reference, top_db=None)
print("Размер матрицы (частоты, участки):", D_noise.shape)
print(f"Длина окна: {1000 * n_fft / sr:.1f} мс; шаг по частоте: {sr / n_fft:.1f} Гц")
for title, matrix in [("Исходник", clean_db), ("С шумом", noise_db)]:
    plt.figure(figsize=(10, 2.8))
    image = librosa.display.specshow(matrix, sr=sr, hop_length=hop_length,
                                    x_axis="time", y_axis="linear", vmin=-80, vmax=0)
    plt.xlabel("Время, с")
    plt.ylabel("Частота, Гц")
    plt.title(title)
    plt.colorbar(image, label="Уровень, дБ")
    plt.tight_layout()
    plt.show()
listen("Записи для спектрограмм", [("Исходник", fragment, sr), ("С шумом", noisy, sr)])

Размер матрицы (частоты, участки): (513, 751)
Длина окна: 42.7 мс; шаг по частоте: 23.4 Гц


### Спектральное отсечение и восстановление
В копии матрицы обнулим слабые коэффициенты и частоты выше выбранной границы. `threshold_db` задает порог уровня, `cutoff_hz` задает верхнюю частоту. Вместе с шумом могут исчезнуть компоненты речи или музыки.

`librosa.istft` восстанавливает запись из комплексных коэффициентов; `length` сохраняет число отсчетов. На рисунке показана спектрограмма восстановленного звука.

In [ ]:
#@title Отсечение компонентов { display-mode: "both", run: "auto" }
recording = "Речь" #@param ["Речь", "Мелодия"]
noise_std = 0.04 #@param {type:"slider", min:0, max:0.12, step:0.01}
cutoff_hz = 4000 #@param {type:"slider", min:500, max:12000, step:500}
threshold_db = -35 #@param {type:"slider", min:-60, max:-10, step:5}
n_fft = 1024 #@param [512, 1024, 2048] {type:"raw"}

signal, sr = recordings[recording]
fragment = signal[:8 * sr]
noisy = fragment + np.random.default_rng(42).normal(0, noise_std, len(fragment))
hop_length = n_fft // 4
D_clean = librosa.stft(fragment, n_fft=n_fft, hop_length=hop_length)
D_noise = librosa.stft(noisy, n_fft=n_fft, hop_length=hop_length)
reference = max(np.abs(D_clean).max(), np.abs(D_noise).max()) or 1.0
noise_db = librosa.amplitude_to_db(np.abs(D_noise), ref=reference, top_db=None)
frequencies = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

# Изменяем копию комплексной матрицы. Фаза сохраненных коэффициентов остается.
D_filtered = D_noise.copy()
D_filtered[noise_db < threshold_db] = 0
D_filtered[frequencies > cutoff_hz, :] = 0
restored = librosa.istft(D_filtered, hop_length=hop_length, length=len(fragment))
print("Отсчетов до и после:", len(fragment), len(restored))

D_after = librosa.stft(restored, n_fft=n_fft, hop_length=hop_length)
after_db = librosa.amplitude_to_db(np.abs(D_after), ref=reference, top_db=None)
plt.figure(figsize=(10, 2.8))
image = librosa.display.specshow(after_db, sr=sr, hop_length=hop_length,
                                x_axis="time", y_axis="linear", vmin=-80, vmax=0)
plt.xlabel("Время, с")
plt.ylabel("Частота, Гц")
plt.title("После отсечения")
plt.colorbar(image, label="Уровень, дБ")
plt.tight_layout()
plt.show()
listen("Результат обработки", [
    ("Исходник", fragment, sr), ("С шумом", noisy, sr), ("После отсечения", restored, sr),
])

Отсчетов до и после: 192000 192000


## Домашнее задание
Собери запись «вперед, назад, вперед». Возьми 0,5–2 секунды речи или мелодии и допиши три строки: выдели фрагмент в `piece`, разверни его в `backwards`, соедини обычный, обратный и снова обычный фрагменты в `result`.

Начало и конец выбираются в секундах.

In [ ]:
recording = "Речь"  # Можно заменить на "Мелодия".
start_s = 1.0
end_s = 2.0
signal, sr = recordings[recording]

piece = None       # Замени None срезом массива signal.
backwards = None   # Замени None разворотом piece.
result = None      # Замени None соединением трех массивов.

# Этот фрагмент оставь без изменений.
if result is None:
    print("Допиши три строки выше и запусти ячейку.")
else:
    output_path = RESULT_DIR / "nk-p01-dan-domashka.wav"
    sf.write(output_path, result, sr, subtype="FLOAT")
    print("Файл:", output_path)
    print("Длительность:", round(len(result) / sr, 2), "с")
    listen("Домашняя работа", [("Вперед, назад, вперед", result, sr)])

Допиши три строки выше и запусти ячейку.


## Справка для домашнего задания
### Переменная и присваивание
Справа от `=` находится значение или вычисление, слева имя, под которым сохраняется результат.

```python
seconds = 2.5
rate = 8_000
count = int(seconds * rate)
print(count)  # 20000
```

Дробные числа пишутся через точку. `int` преобразует число в целое; номера отсчетов должны быть целыми. Строки, например `"Речь"`, пишутся в кавычках.

### Позиции и срезы
Нумерация начинается с нуля. Правая граница среза не включается.

```python
values = np.array([10, 20, 30, 40, 50, 60])
print(values[0])     # 10
print(values[1:4])   # [20 30 40]
print(values[:3])    # [10 20 30]
print(values[3:])    # [40 50 60]
```

Для звука граница в секунду `t` соответствует позиции `int(t * sr)`. `len(signal)` дает число отсчетов всей записи.

### Обратный порядок
Третий параметр среза задает шаг. Шаг `-1` читает массив с конца.

```python
values = np.array([10, 20, 30, 40])
print(values[::-1])  # [40 30 20 10]
```

Разворот не изменяет число отсчетов. При прежнем `sr` длительность сохраняется.

### Соединение
`np.concatenate` получает список массивов и ставит их друг за другом. Порядок в квадратных скобках задает порядок в результате.

```python
first = np.array([10, 20])
second = np.array([30, 40])
joined = np.concatenate([second, first])
print(joined)  # [30 40 10 20]
```

Оператор `+` для массивов одинаковой длины складывает соответствующие числа. Для последовательного воспроизведения нескольких частей нужен `np.concatenate`.

### Если ячейка выдала ошибку
`NameError` означает, что имя не определено: проверь написание и запуск предыдущих ячеек. `SyntaxError` указывает на ошибку записи: проверь кавычки, запятые и скобки. Срезы пишутся в квадратных скобках, вызовы функций в круглых.

Если результат пустой, проверь, что начало меньше конца и обе границы находятся внутри записи. Ее длительность можно узнать командой `len(signal) / sr`. После изменения кода домашней работы запусти ее ячейку еще раз.

### Файл и проигрыватель
`sf.write(output_path, result, sr, subtype="FLOAT")` сохраняет результат в WAV. Файл `nk-p01-dan-domashka.wav` находится в папке `nk-p01-results`; его можно скачать через панель «Файлы» Colab.

`listen` открывает Gradio для переданных массивов. При прослушивании используется одинаковый коэффициент уровня `0.35`; сохраненный файл домашней работы содержит исходные значения `result`. Автоматического усиления каждого варианта отдельно нет.

### Документация
[Срезы NumPy](https://numpy.org/doc/stable/user/basics.indexing.html), [соединение массивов](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html), [чтение и запись soundfile](https://python-soundfile.readthedocs.io/), [формы Colab](https://colab.research.google.com/notebooks/forms.ipynb).